In [ ]:
import pandas as pd
df = pd.read_csv("/content/102_Nutrients_texual.csv")

In [ ]:
df["text"] = (
    df["Main_food_description"].fillna("")
    + " "
    + df["catname"].fillna("")
    + " "
    + df["macroclass"].fillna("")
)

In [ ]:
# Generate embeddings

MODEL_NAME = "distilbert-base-uncased"

from transformers import AutoTokenizer
from transformers import AutoModel

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModel.from_pretrained(
    MODEL_NAME
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#Checks if device has GPU for faster computation
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model.to(device)
model.eval()

print(device)

cuda


In [ ]:
# Embedding extraction

import torch
import numpy as np
from tqdm import tqdm

#taking batches of 32 at a time
def generate_embeddings_batch(
    texts,
    batch_size=32
):

    all_embeddings = []

    for i in tqdm(
        range(0, len(texts), batch_size)
    ):

        batch_texts = texts[
            i : i + batch_size
        ]

        tokens = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        tokens = {
            k: v.to(device)
            for k, v in tokens.items()
        }

        with torch.no_grad():

            outputs = model(**tokens)

            embeddings = (
                outputs.last_hidden_state
                .mean(dim=1)
                .cpu()
                .numpy()
            )

        all_embeddings.append(
            embeddings
        )

    return np.vstack(
        all_embeddings
    )

In [ ]:
# Create embedding matrix

embeddings = generate_embeddings_batch(
    df["text"].tolist(),
    batch_size=32
)

100%|██████████| 93/93 [00:03<00:00, 29.71it/s]


In [ ]:
np.save(
    "xlmr_embeddings.npy",
    embeddings
)

print(
    "saved",
    embeddings.shape
)

saved (2970, 768)


In [ ]:
# Create nutrient matrix

meta_cols = [
"Food_code",
"Main_food_description",
"catnumb",
"catname",
"novaclass",
"macroclass",
"text"]

nutrient_cols = [
    col
    for col in df.columns
    if col not in meta_cols
]

X_nutrients = df[nutrient_cols].values

In [ ]:
X_nutrients.shape # (2970,102)

(2970, 103)

In [ ]:
# target

y = df["novaclass"]

In [ ]:
# cross-validation
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import numpy as np

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Define model
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

print("\nTraining Random Forest...\n")

# Lists to store metrics
acc_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
mcc_scores = []

# Cross-validation
for train_idx, test_idx in skf.split(
    X_nutrients,
    y_encoded
):

    # Split labels
    y_train = y_encoded[train_idx]
    y_test = y_encoded[test_idx]

    # Nutrient features
    X_train_nutrient = X_nutrients[train_idx, :]
    X_test_nutrient = X_nutrients[test_idx, :]

    # Scale nutrients
    scaler = StandardScaler()

    X_train_nutrient = scaler.fit_transform(
        X_train_nutrient
    )

    X_test_nutrient = scaler.transform(
        X_test_nutrient
    )

    # DistilBERT embeddings
    X_train_embed = embeddings[train_idx]
    X_test_embed = embeddings[test_idx]

    # Combine features
    X_train = np.hstack(
        (
            X_train_nutrient,
            X_train_embed
        )
    )

    X_test = np.hstack(
        (
            X_test_nutrient,
            X_test_embed
        )
    )

    # Apply SMOTE
    smote = SMOTE(
        random_state=42
    )

    X_train_resampled, y_train_resampled = (
        smote.fit_resample(
            X_train,
            y_train
        )
    )

    # Train model
    model.fit(
        X_train_resampled,
        y_train_resampled
    )

    # Predict
    y_pred = model.predict(
        X_test
    )

    # Store metrics
    acc_scores.append(
        accuracy_score(
            y_test,
            y_pred
        )
    )

    precision_scores.append(
        precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    recall_scores.append(
        recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    f1_scores.append(
        f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    mcc_scores.append(
        matthews_corrcoef(
            y_test,
            y_pred
        )
    )

# Final results
print("\nFINAL RESULTS\n")

print("Accuracy :", np.mean(acc_scores))
print("Precision:", np.mean(precision_scores))
print("Recall   :", np.mean(recall_scores))
print("F1 Score :", np.mean(f1_scores))
print("MCC      :", np.mean(mcc_scores))


Training Random Forest...


FINAL RESULTS

Accuracy : 0.934006734006734
Precision: 0.9348283603581597
Recall   : 0.934006734006734
F1 Score : 0.933624469101
MCC      : 0.855085491775116


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import numpy as np

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Define model
model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

print("\nTraining Random Forest...\n")

# Lists to store metrics
acc_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
mcc_scores = []

# Cross-validation
for train_idx, test_idx in skf.split(
    X_nutrients,
    y_encoded
):

    # Split labels
    y_train = y_encoded[train_idx]
    y_test = y_encoded[test_idx]

    # Nutrient features
    X_train_nutrient = X_nutrients[train_idx, :]
    X_test_nutrient = X_nutrients[test_idx, :]

    # Scale nutrients
    scaler = StandardScaler()

    X_train_nutrient = scaler.fit_transform(
        X_train_nutrient
    )

    X_test_nutrient = scaler.transform(
        X_test_nutrient
    )

    # DistilBERT embeddings
    X_train_embed = embeddings[train_idx]
    X_test_embed = embeddings[test_idx]

    # Combine features
    X_train = np.hstack(
        (
            X_train_nutrient,
            X_train_embed
        )
    )

    X_test = np.hstack(
        (
            X_test_nutrient,
            X_test_embed
        )
    )

    # Apply SMOTE
    smote = SMOTE(
        random_state=42
    )

    X_train_resampled, y_train_resampled = (
        smote.fit_resample(
            X_train,
            y_train
        )
    )

    # Train model
    model.fit(
        X_train_resampled,
        y_train_resampled
    )

    # Predict
    y_pred = model.predict(
        X_test
    )

    # Store metrics
    acc_scores.append(
        accuracy_score(
            y_test,
            y_pred
        )
    )

    precision_scores.append(
        precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    recall_scores.append(
        recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    f1_scores.append(
        f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    mcc_scores.append(
        matthews_corrcoef(
            y_test,
            y_pred
        )
    )

# Final results
print("\nFINAL RESULTS\n")

print("Accuracy :", np.mean(acc_scores))
print("Precision:", np.mean(precision_scores))
print("Recall   :", np.mean(recall_scores))
print("F1 Score :", np.mean(f1_scores))
print("MCC      :", np.mean(mcc_scores))


Training Random Forest...


FINAL RESULTS

Accuracy : 0.9329966329966328
Precision: 0.9331003879931913
Recall   : 0.9329966329966328
F1 Score : 0.9323493758739001
MCC      : 0.8520192705809635


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import numpy as np

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Define model
model = XGBClassifier(
    random_state=42,
    eval_metric="mlogloss"
)

print("\nTraining Random Forest...\n")

# Lists to store metrics
acc_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
mcc_scores = []

# Cross-validation
for train_idx, test_idx in skf.split(
    X_nutrients,
    y_encoded
):

    # Split labels
    y_train = y_encoded[train_idx]
    y_test = y_encoded[test_idx]

    # Nutrient features
    X_train_nutrient = X_nutrients[train_idx, :]
    X_test_nutrient = X_nutrients[test_idx, :]

    # Scale nutrients
    scaler = StandardScaler()

    X_train_nutrient = scaler.fit_transform(
        X_train_nutrient
    )

    X_test_nutrient = scaler.transform(
        X_test_nutrient
    )

    # DistilBERT embeddings
    X_train_embed = embeddings[train_idx]
    X_test_embed = embeddings[test_idx]

    # Combine features
    X_train = np.hstack(
        (
            X_train_nutrient,
            X_train_embed
        )
    )

    X_test = np.hstack(
        (
            X_test_nutrient,
            X_test_embed
        )
    )

    # Apply SMOTE
    smote = SMOTE(
        random_state=42
    )

    X_train_resampled, y_train_resampled = (
        smote.fit_resample(
            X_train,
            y_train
        )
    )

    # Train model
    model.fit(
        X_train_resampled,
        y_train_resampled
    )

    # Predict
    y_pred = model.predict(
        X_test
    )

    # Store metrics
    acc_scores.append(
        accuracy_score(
            y_test,
            y_pred
        )
    )

    precision_scores.append(
        precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    recall_scores.append(
        recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    f1_scores.append(
        f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    mcc_scores.append(
        matthews_corrcoef(
            y_test,
            y_pred
        )
    )

# Final results
print("\nFINAL RESULTS\n")

print("Accuracy :", np.mean(acc_scores))
print("Precision:", np.mean(precision_scores))
print("Recall   :", np.mean(recall_scores))
print("F1 Score :", np.mean(f1_scores))
print("MCC      :", np.mean(mcc_scores))


Training Random Forest...


FINAL RESULTS

Accuracy : 0.946127946127946
Precision: 0.9472861113214682
Recall   : 0.946127946127946
F1 Score : 0.946073165778758
MCC      : 0.8829819468766603


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import numpy as np

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Define model
model = LGBMClassifier(random_state=42)

print("\nTraining Random Forest...\n")

# Lists to store metrics
acc_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
mcc_scores = []

# Cross-validation
for train_idx, test_idx in skf.split(
    X_nutrients,
    y_encoded
):

    # Split labels
    y_train = y_encoded[train_idx]
    y_test = y_encoded[test_idx]

    # Nutrient features
    X_train_nutrient = X_nutrients[train_idx, :]
    X_test_nutrient = X_nutrients[test_idx, :]

    # Scale nutrients
    scaler = StandardScaler()

    X_train_nutrient = scaler.fit_transform(
        X_train_nutrient
    )

    X_test_nutrient = scaler.transform(
        X_test_nutrient
    )

    # DistilBERT embeddings
    X_train_embed = embeddings[train_idx]
    X_test_embed = embeddings[test_idx]

    # Combine features
    X_train = np.hstack(
        (
            X_train_nutrient,
            X_train_embed
        )
    )

    X_test = np.hstack(
        (
            X_test_nutrient,
            X_test_embed
        )
    )

    # Apply SMOTE
    smote = SMOTE(
        random_state=42
    )

    X_train_resampled, y_train_resampled = (
        smote.fit_resample(
            X_train,
            y_train
        )
    )

    # Train model
    model.fit(
        X_train_resampled,
        y_train_resampled
    )

    # Predict
    y_pred = model.predict(
        X_test
    )

    # Store metrics
    acc_scores.append(
        accuracy_score(
            y_test,
            y_pred
        )
    )

    precision_scores.append(
        precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    recall_scores.append(
        recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    f1_scores.append(
        f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    mcc_scores.append(
        matthews_corrcoef(
            y_test,
            y_pred
        )
    )

# Final results
print("\nFINAL RESULTS\n")

print("Accuracy :", np.mean(acc_scores))
print("Precision:", np.mean(precision_scores))
print("Recall   :", np.mean(recall_scores))
print("F1 Score :", np.mean(f1_scores))
print("MCC      :", np.mean(mcc_scores))


Training Random Forest...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.164483 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 217251
[LightGBM] [Info] Number of data points in the train set: 6760, number of used features: 871
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [War

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.081513 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 217284
[LightGBM] [Info] Number of data points in the train set: 6760, number of used features: 871
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.074316 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 217201
[LightGBM] [Info] Number of data points in the train set: 6760, number of used features: 871
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 217145
[LightGBM] [Info] Number of data points in the train set: 6756, number of used features: 871
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.133258 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 217306
[LightGBM] [Info] Number of data points in the train set: 6756, number of used features: 871
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
